## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        
        # host='presto.processing.yoda.run',
        # host='presto-gateway.serving.data.production.internal',
        # host="bi-presto.serving.data.production.internal",
        # host="insights-download.serving.data.production.internal",
        # host="bi-trino-2.serving.data.production.internal",
        # host="bi-trino.serving.data.production.internal",
        # host="presto.processing.yoda.run",
        host='presto-gateway.processing.data.production.internal',
        # host="processing-2.processing.data.production.internal",
        # host="processing.processing.data.production.internal",
    
    
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [31]:
## Parameter 
start_date = '20240423'
end_date = '20240424'

# ('Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Jaipur', 'Pune', 'Kolkata', 'Ahmedabad', 'Indore', 'Vijayawada', 'Chandigarh', 'Lucknow', 'Mumbai', 'Coimbatore', 'Guwahati')
# ('Auto', 'Auto Pool', 'AutoPremium', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'Cab Premium', 'CabEconomy', 'CityAuto', 'Link')

### customer_postorderstatus_started_immutable

In [32]:
### experiment_user_base

experiment_user_base = f"""

    with experiment_user_base as (
    
    select
        userId,
        max(pageVariant) pageVariant
    from
        (
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_ontheway_immutable
        where 
            yyyymmdd >= '{start_date}'
            and yyyymmdd <= '{end_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') IN ('Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Jaipur', 'Pune', 'Kolkata' )
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_arrived_immutable
        where 
            yyyymmdd >= '{start_date}'
            and yyyymmdd <= '{end_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') IN ('Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Jaipur', 'Pune', 'Kolkata' )
        
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_started_immutable
        where 
            yyyymmdd >= '{start_date}'
            and yyyymmdd <= '{end_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') IN ('Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Jaipur', 'Pune', 'Kolkata' )
        )
    group by 1
    ),
    
    orderLogsFact as (

    select
        yyyymmdd,
        service_obj_service_name service_name,
        customer_id,
        city_name,
        order_id,
        order_status,
        modified_order_status,
        spd_fraud_flag,
        CASE WHEN modified_order_status in ('OCARA', 'COBRA', 'COBRM') THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END time_spent_in_sec,
        channel_host
    from
        orders.order_logs_fact
    where 
        yyyymmdd >= '{start_date}'
        and yyyymmdd <= '{end_date}'
        and city_name IN ('Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Jaipur', 'Pune', 'Kolkata' )
        and service_obj_service_name IN ('Auto', 'Auto Pool', 'AutoPremium', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'Cab Premium', 'CabEconomy', 'CityAuto', 'Link')
    )
    
    
    
    select 
        pageVariant,
        yyyymmdd,
        customer_id,
        channel_host,
        city_name,
        service_name,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end) net_orders,
        count(distinct case when modified_order_status = 'COBRM' then order_id end) cobrm,
        count(distinct case when modified_order_status = 'COBRA' then order_id end) cobra,
        count(distinct case when modified_order_status = 'OCARA' then order_id end) ocara,
        count(distinct case when modified_order_status = 'aborted' then order_id end) aborted,
        count(distinct case when modified_order_status = 'expired' then order_id end) expired,
        
        avg(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END) as avg_cobrm_time,
        avg(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END) as avg_cobra_time,
        avg(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END) as avg_ocara_time
        
    from 
        experiment_user_base
    join 
        orderLogsFact
        on userId = customer_id
    group by 1,2,3,4,5,6
"""

In [33]:
df_experiment_user_base = pd.read_sql(experiment_user_base, connection)

# Local
df_experiment_user_base.to_csv('df_experiment_user_base1.csv', index=False)
df_experiment_user_base = pd.read_csv('df_experiment_user_base1.csv')
df_experiment_user_base.head(5)

,pageVariant,yyyymmdd,customer_id,channel_host,city_name,service_name,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,avg_cobrm_time,avg_cobra_time,avg_ocara_time
0,l1Cancel,20240424,61a1d6b7423d1818e0ac2af2,android,Hyderabad,Auto,2,0,0,0,2,0,0,NaN,NaN,95.0
1,l1Cancel,20240423,61a1d6b7423d1818e0ac2af2,android,Hyderabad,Auto,3,2,0,0,1,0,0,NaN,NaN,27.0
2,l2Cancel,20240423,6417c48ef6a73c282e8fd02e,android,Bangalore,Auto,1,1,0,0,0,0,0,NaN,NaN,NaN
3,l2Cancel,20240424,6417c48ef6a73c282e8fd02e,android,Bangalore,Auto,1,1,0,0,0,0,0,NaN,NaN,NaN
4,l1Cancel,20240424,643502c23c2e3c7ac234416b,android,Delhi,Bike Lite,3,0,1,2,0,0,0,41.0,96.0,NaN


In [34]:
df_experiment_user_base\
.groupby(['pageVariant', 'channel_host', 'city_name'])\
.agg(customers = ('customer_id', 'nunique'))

customers
pageVariant channel_host city_name           
l1Cancel    android      Bangalore       7407
                         Chennai       195276
                         Delhi         128229
                         Hyderabad     409388
                         Jaipur         27667
                         Kolkata         2059
                         Pune           33140
            ios          Bangalore          8
                         Chennai           39
                         Delhi             61
                         Hyderabad        113
                         Jaipur             4
                         Pune               3
l2Cancel    android      Bangalore     266712
                         Chennai        19809
                         Delhi         152076
                         Hyderabad      45759
                         Jaipur         31472
                         Kolkata        52876
                         Pune            3403
            ios          Bangalore         88
                         Chennai            5
                         Delhi             60
                         Hyderabad         13
                         Jaipur             8
                         Kolkata            8
                         Pune               1

### iceberg_log_telemetry_ads_impressions_immutable

In [35]:
## telemetry_ads

telemetry_ads = f"""

with olf_data as (

        select
            yyyymmdd yyyymd,
            customer_id,
            channel_host,
            city_name
        from
            orders.order_logs_fact
        where 
            yyyymmdd >= '{start_date}'
            and yyyymmdd <= '{end_date}'
            and city_name in ('Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Jaipur', 'Pune', 'Kolkata')
            and service_obj_service_name in ('Auto', 'Auto Pool', 'AutoPremium', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'Cab Premium', 'CabEconomy', 'CityAuto', 'Link')
        group by 1,2,3,4
        ), 

    telemetry_ads AS (

    SELECT
                
        yyyymmdd,
        userId,
        os,
        city_name,
        COUNT(DISTINCT CASE WHEN eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS ad_request,
        COUNT(DISTINCT CASE WHEN eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS ad_response,
        COUNT(DISTINCT CASE WHEN eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS ad_viewable_impression,
        COUNT(DISTINCT CASE WHEN eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS ad_click,

        COUNT(DISTINCT CASE WHEN pagename = 'PostOrderScreen' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS pos_ad_request,
        COUNT(DISTINCT CASE WHEN pagename = 'PostOrderScreen' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS pos_ad_response,
        COUNT(DISTINCT CASE WHEN pagename = 'PostOrderScreen' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS pos_ad_viewable_impression,
        COUNT(DISTINCT CASE WHEN pagename = 'PostOrderScreen' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS pos_ad_click,

        COUNT(DISTINCT CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS dispatch_ad_request,
        COUNT(DISTINCT CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS dispatch_ad_response,
        COUNT(DISTINCT CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS dispatch_ad_viewable_impression,
        COUNT(DISTINCT CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS dispatch_ad_click,

        COUNT(DISTINCT CASE WHEN screen_slot_name = 'onTheWay' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS ontheway_ad_request,
        COUNT(DISTINCT CASE WHEN screen_slot_name = 'onTheWay' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS ontheway_ad_response,
        COUNT(DISTINCT CASE WHEN screen_slot_name = 'onTheWay' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS ontheway_ad_viewable_impression,
        COUNT(DISTINCT CASE WHEN screen_slot_name = 'onTheWay' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS ontheway_ad_click,

        COUNT(DISTINCT CASE WHEN screen_slot_name = 'arrived' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS arrived_ad_request,
        COUNT(DISTINCT CASE WHEN screen_slot_name = 'arrived' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS arrived_ad_response,
        COUNT(DISTINCT CASE WHEN screen_slot_name = 'arrived' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS arrived_ad_viewable_impression,
        COUNT(DISTINCT CASE WHEN screen_slot_name = 'arrived' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS arrived_ad_click,

        COUNT(DISTINCT CASE WHEN screen_slot_name = 'started' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS started_ad_request,
        COUNT(DISTINCT CASE WHEN screen_slot_name = 'started' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS started_ad_response,
        COUNT(DISTINCT CASE WHEN screen_slot_name = 'started' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS started_ad_viewable_impression,
        COUNT(DISTINCT CASE WHEN screen_slot_name = 'started' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS started_ad_click,

        COUNT(DISTINCT CASE WHEN pagename = 'HomeScreen' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS home_ad_request,
        COUNT(DISTINCT CASE WHEN pagename = 'HomeScreen' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS home_ad_response,
        COUNT(DISTINCT CASE WHEN pagename = 'HomeScreen' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END ) AS home_ad_viewable_impression,
        COUNT(DISTINCT CASE WHEN pagename = 'HomeScreen' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END) AS home_ad_click
        
    FROM   
        (
        SELECT 
            yyyymmdd,
            userId,
            os,
            city_name,
            pagename,
            CASE 
            WHEN pagename IN ('HomeScreen', 'CaptainSearchScreen') THEN pagename
            WHEN ltrim(orderstatus) IS NOT NULL THEN orderstatus
            WHEN lower(slotname) LIKE '%ontheway%' THEN 'onTheWay'
            WHEN lower(slotname) LIKE '%arrived%' THEN 'arrived'
            WHEN lower(slotname) LIKE '%started%' THEN 'started'
            END screen_slot_name,
            eventName,
            hhmmss
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable 
        join 
            olf_data
            on yyyymmdd = yyyymd 
            and customer_id = userId    
            and channel_host = os
        WHERE  
            yyyymmdd >= '{start_date}'
            and yyyymmdd <= '{end_date}'
            and responseType = 'GAMBanner'
        )
    GROUP BY 1,2,3,4
    )
    
    SELECT 
        DISTINCT *
    FROM 
        telemetry_ads
"""

In [36]:
df_telemetry_ads = pd.read_sql(telemetry_ads, connection)

In [37]:

## Local
df_telemetry_ads.to_csv('df_telemetry_ads1.csv', index=False)
df_telemetry_ads = pd.read_csv('df_telemetry_ads1.csv')
df_telemetry_ads.head(5)

,yyyymmdd,userId,os,city_name,ad_request,ad_response,ad_viewable_impression,ad_click,pos_ad_request,pos_ad_response,pos_ad_viewable_impression,pos_ad_click,dispatch_ad_request,dispatch_ad_response,dispatch_ad_viewable_impression,dispatch_ad_click,ontheway_ad_request,ontheway_ad_response,ontheway_ad_viewable_impression,ontheway_ad_click,arrived_ad_request,arrived_ad_response,arrived_ad_viewable_impression,arrived_ad_click,started_ad_request,started_ad_response,started_ad_viewable_impression,started_ad_click,home_ad_request,home_ad_response,home_ad_viewable_impression,home_ad_click
0,20240424,657089799e30ffb27428159c,android,Bangalore,18,9,0,0,12,9,0,0,6,0,0,0,9,5,0,0,2,2,0,0,1,2,0,0,0,0,0,0
1,20240424,5f3ffd44d358d1b7954fc518,android,Hyderabad,16,4,0,0,14,4,0,0,2,0,0,0,5,4,0,0,2,0,0,0,7,0,0,0,0,0,0,0
2,20240424,62dcdf2272478a1d0e9d75c4,android,Jaipur,15,3,0,0,9,3,0,0,6,0,0,0,9,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,20240424,5d078c6f57152c471e015f18,android,Hyderabad,31,1,0,0,18,1,0,0,13,0,0,0,6,1,0,0,4,0,0,0,8,0,0,0,0,0,0,0
4,20240424,64313d7f75499b4197ab2640,android,Delhi,16,5,4,0,10,5,4,0,6,0,0,0,5,4,3,0,2,0,0,0,3,1,1,0,0,0,0,0


## User Define Function

In [38]:
def calculate_percentage(df, numerator, denominator):
    percentage = (df[numerator] * 100.00 / df[denominator]).round(2)
    return percentage

In [39]:
df_experiment_user_base.head(2)

,pageVariant,yyyymmdd,customer_id,channel_host,city_name,service_name,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,avg_cobrm_time,avg_cobra_time,avg_ocara_time
0,l1Cancel,20240424,61a1d6b7423d1818e0ac2af2,android,Hyderabad,Auto,2,0,0,0,2,0,0,NaN,NaN,95.0
1,l1Cancel,20240423,61a1d6b7423d1818e0ac2af2,android,Hyderabad,Auto,3,2,0,0,1,0,0,NaN,NaN,27.0


## Merge

In [40]:
df_experiment_user_base.head(2)

,pageVariant,yyyymmdd,customer_id,channel_host,city_name,service_name,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,avg_cobrm_time,avg_cobra_time,avg_ocara_time
0,l1Cancel,20240424,61a1d6b7423d1818e0ac2af2,android,Hyderabad,Auto,2,0,0,0,2,0,0,NaN,NaN,95.0
1,l1Cancel,20240423,61a1d6b7423d1818e0ac2af2,android,Hyderabad,Auto,3,2,0,0,1,0,0,NaN,NaN,27.0


In [41]:
df_telemetry_ads.head(2)

,yyyymmdd,userId,os,city_name,ad_request,ad_response,ad_viewable_impression,ad_click,pos_ad_request,pos_ad_response,pos_ad_viewable_impression,pos_ad_click,dispatch_ad_request,dispatch_ad_response,dispatch_ad_viewable_impression,dispatch_ad_click,ontheway_ad_request,ontheway_ad_response,ontheway_ad_viewable_impression,ontheway_ad_click,arrived_ad_request,arrived_ad_response,arrived_ad_viewable_impression,arrived_ad_click,started_ad_request,started_ad_response,started_ad_viewable_impression,started_ad_click,home_ad_request,home_ad_response,home_ad_viewable_impression,home_ad_click
0,20240424,657089799e30ffb27428159c,android,Bangalore,18,9,0,0,12,9,0,0,6,0,0,0,9,5,0,0,2,2,0,0,1,2,0,0,0,0,0,0
1,20240424,5f3ffd44d358d1b7954fc518,android,Hyderabad,16,4,0,0,14,4,0,0,2,0,0,0,5,4,0,0,2,0,0,0,7,0,0,0,0,0,0,0


In [42]:
## Filter by experiment_user_base

df_base_final = pd.merge(df_experiment_user_base,
                    df_telemetry_ads,
                    how = 'left',
                    left_on = ['yyyymmdd', 'customer_id','channel_host', 'city_name'],
                    right_on = ['yyyymmdd', 'userId', 'os', 'city_name'],
                   )

df_base_final.head(5)

,pageVariant,yyyymmdd,customer_id,channel_host,city_name,service_name,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,avg_cobrm_time,avg_cobra_time,avg_ocara_time,userId,os,ad_request,ad_response,ad_viewable_impression,ad_click,pos_ad_request,pos_ad_response,pos_ad_viewable_impression,pos_ad_click,dispatch_ad_request,dispatch_ad_response,dispatch_ad_viewable_impression,dispatch_ad_click,ontheway_ad_request,ontheway_ad_response,ontheway_ad_viewable_impression,ontheway_ad_click,arrived_ad_request,arrived_ad_response,arrived_ad_viewable_impression,arrived_ad_click,started_ad_request,started_ad_response,started_ad_viewable_impression,started_ad_click,home_ad_request,home_ad_response,home_ad_viewable_impression,home_ad_click
0,l1Cancel,20240424,61a1d6b7423d1818e0ac2af2,android,Hyderabad,Auto,2,0,0,0,2,0,0,NaN,NaN,95.0,61a1d6b7423d1818e0ac2af2,android,9.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,l1Cancel,20240423,61a1d6b7423d1818e0ac2af2,android,Hyderabad,Auto,3,2,0,0,1,0,0,NaN,NaN,27.0,61a1d6b7423d1818e0ac2af2,android,24.0,2.0,0.0,0.0,19.0,2.0,0.0,0.0,5.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,8.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
2,l2Cancel,20240423,6417c48ef6a73c282e8fd02e,android,Bangalore,Auto,1,1,0,0,0,0,0,NaN,NaN,NaN,6417c48ef6a73c282e8fd02e,android,13.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,l2Cancel,20240424,6417c48ef6a73c282e8fd02e,android,Bangalore,Auto,1,1,0,0,0,0,0,NaN,NaN,NaN,6417c48ef6a73c282e8fd02e,android,7.0,1.0,0.0,0.0,5.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,l1Cancel,20240424,643502c23c2e3c7ac234416b,android,Delhi,Bike Lite,3,0,1,2,0,0,0,41.0,96.0,NaN,643502c23c2e3c7ac234416b,android,22.0,4.0,0.0,0.0,14.0,4.0,0.0,0.0,8.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,5.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0


## Analysis 

In [43]:
df = df_base_final\
.groupby(['pageVariant', 'yyyymmdd'])\
.agg(gross_orders = ('gross_orders', 'sum'),
     net_orders = ('net_orders', 'sum'),
     cobrm = ('cobrm', 'sum'),
     cobra = ('cobra', 'sum'),
     ocara = ('ocara', 'sum'),
     aborted = ('aborted', 'sum'),
     expired = ('expired', 'sum'),
     median_cobrm_time = ('avg_cobrm_time', lambda x: x.quantile(0.50)),
     median_cobra_time = ('avg_cobra_time', lambda x: x.quantile(0.50)),
     median_ocara_time = ('avg_ocara_time', lambda x: x.quantile(0.50)),
     
     ad_request = ('ad_request', 'sum'),
     ad_response = ('ad_response', 'sum'),
     ad_viewable_impression = ('ad_viewable_impression', 'sum'),
     ad_click = ('ad_click', 'sum'),
     
     pos_ad_request = ('pos_ad_request', 'sum'),
     pos_ad_response = ('pos_ad_response', 'sum'),
     pos_ad_viewable_impression = ('pos_ad_viewable_impression', 'sum'),
     pos_ad_click = ('pos_ad_click', 'sum'),
     
     ontheway_ad_request = ('ontheway_ad_request', 'sum'),
     ontheway_ad_response = ('ontheway_ad_response', 'sum'),
     ontheway_ad_viewable_impression = ('ontheway_ad_viewable_impression', 'sum'),
     ontheway_ad_click = ('ontheway_ad_click', 'sum'),
     
     arrived_ad_request = ('arrived_ad_request', 'sum'),
     arrived_ad_response = ('arrived_ad_response', 'sum'),
     arrived_ad_viewable_impression = ('arrived_ad_viewable_impression', 'sum'),
     arrived_ad_click = ('arrived_ad_click', 'sum'),
     
     started_ad_request = ('started_ad_request', 'sum'),
     started_ad_response = ('started_ad_response', 'sum'),
     started_ad_viewable_impression = ('started_ad_viewable_impression', 'sum'),
     started_ad_click = ('started_ad_click', 'sum'),
    )\
.reset_index()

In [44]:
df

,pageVariant,yyyymmdd,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,median_cobrm_time,median_cobra_time,median_ocara_time,ad_request,ad_response,ad_viewable_impression,ad_click,pos_ad_request,pos_ad_response,pos_ad_viewable_impression,pos_ad_click,ontheway_ad_request,ontheway_ad_response,ontheway_ad_viewable_impression,ontheway_ad_click,arrived_ad_request,arrived_ad_response,arrived_ad_viewable_impression,arrived_ad_click,started_ad_request,started_ad_response,started_ad_viewable_impression,started_ad_click
0,l1Cancel,20240423,912752,513429,7655,149383,166197,1264,71245,112.0,139.0,333.416667,6753585.0,1523642.0,257666.0,3124.0,4391778.0,1523642.0,257666.0,3124.0,2110074.0,705393.0,134935.0,1903.0,977883.0,314661.0,38048.0,150.0,1306789.0,503588.0,84683.0,1071.0
1,l1Cancel,20240424,868090,480849,7146,155671,146196,1042,74091,111.0,142.5,334.000000,6262235.0,1452449.0,323362.0,2616.0,4042771.0,1452449.0,323362.0,2616.0,1911423.0,620375.0,155215.0,1417.0,907840.0,298980.0,56398.0,129.0,1226082.0,533096.0,111749.0,1070.0
2,l2Cancel,20240423,687744,334962,7708,145522,120969,945,75421,96.0,134.6,342.000000,4968989.0,1286707.0,305981.0,3479.0,3057831.0,1286707.0,305981.0,3479.0,1520965.0,629814.0,153110.0,1914.0,658961.0,275310.0,60014.0,198.0,880337.0,381583.0,92858.0,1367.0
3,l2Cancel,20240424,800162,411017,7326,158753,141564,1141,77422,98.0,134.5,345.000000,5963019.0,1448537.0,462549.0,3953.0,3774819.0,1448537.0,462549.0,3953.0,1857948.0,638305.0,205906.0,1903.0,823779.0,312394.0,94057.0,240.0,1096024.0,497839.0,162586.0,1810.0


In [45]:
# df.to_clipboard(index=False)

In [46]:
df2 = df_base_final\
.groupby(['pageVariant'])\
.agg(gross_orders = ('gross_orders', 'sum'),
     net_orders = ('net_orders', 'sum'),
     cobrm = ('cobrm', 'sum'),
     cobra = ('cobra', 'sum'),
     ocara = ('ocara', 'sum'),
     aborted = ('aborted', 'sum'),
     expired = ('expired', 'sum'),
     median_cobrm_time = ('avg_cobrm_time', lambda x: x.quantile(0.50)),
     median_cobra_time = ('avg_cobra_time', lambda x: x.quantile(0.50)),
     median_ocara_time = ('avg_ocara_time', lambda x: x.quantile(0.50)),
     
     ad_request = ('ad_request', 'sum'),
     ad_response = ('ad_response', 'sum'),
     ad_viewable_impression = ('ad_viewable_impression', 'sum'),
     ad_click = ('ad_click', 'sum'),
     
     pos_ad_request = ('pos_ad_request', 'sum'),
     pos_ad_response = ('pos_ad_response', 'sum'),
     pos_ad_viewable_impression = ('pos_ad_viewable_impression', 'sum'),
     pos_ad_click = ('pos_ad_click', 'sum'),
     
     ontheway_ad_request = ('ontheway_ad_request', 'sum'),
     ontheway_ad_response = ('ontheway_ad_response', 'sum'),
     ontheway_ad_viewable_impression = ('ontheway_ad_viewable_impression', 'sum'),
     ontheway_ad_click = ('ontheway_ad_click', 'sum'),
     
     arrived_ad_request = ('arrived_ad_request', 'sum'),
     arrived_ad_response = ('arrived_ad_response', 'sum'),
     arrived_ad_viewable_impression = ('arrived_ad_viewable_impression', 'sum'),
     arrived_ad_click = ('arrived_ad_click', 'sum'),
     
     started_ad_request = ('started_ad_request', 'sum'),
     started_ad_response = ('started_ad_response', 'sum'),
     started_ad_viewable_impression = ('started_ad_viewable_impression', 'sum'),
     started_ad_click = ('started_ad_click', 'sum'),
    )\
.reset_index()

df2

,pageVariant,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,median_cobrm_time,median_cobra_time,median_ocara_time,ad_request,ad_response,ad_viewable_impression,ad_click,pos_ad_request,pos_ad_response,pos_ad_viewable_impression,pos_ad_click,ontheway_ad_request,ontheway_ad_response,ontheway_ad_viewable_impression,ontheway_ad_click,arrived_ad_request,arrived_ad_response,arrived_ad_viewable_impression,arrived_ad_click,started_ad_request,started_ad_response,started_ad_viewable_impression,started_ad_click
0,l1Cancel,1780842,994278,14801,305054,312393,2306,145336,111.333333,140.666667,334.0,13015820.0,2976091.0,581028.0,5740.0,8434549.0,2976091.0,581028.0,5740.0,4021497.0,1325768.0,290150.0,3320.0,1885723.0,613641.0,94446.0,279.0,2532871.0,1036684.0,196432.0,2141.0
1,l2Cancel,1487906,745979,15034,304275,262533,2086,152843,97.000000,134.500000,343.5,10932008.0,2735244.0,768530.0,7432.0,6832650.0,2735244.0,768530.0,7432.0,3378913.0,1268119.0,359016.0,3817.0,1482740.0,587704.0,154071.0,438.0,1976361.0,879422.0,255444.0,3177.0


In [47]:
# df2.to_clipboard(index=False)

In [48]:
df3 = df_base_final\
.groupby(['pageVariant', 'city_name'])\
.agg(gross_orders = ('gross_orders', 'sum'),
     net_orders = ('net_orders', 'sum'),
     cobrm = ('cobrm', 'sum'),
     cobra = ('cobra', 'sum'),
     ocara = ('ocara', 'sum'),
     aborted = ('aborted', 'sum'),
     expired = ('expired', 'sum'),
     median_cobrm_time = ('avg_cobrm_time', lambda x: x.quantile(0.50)),
     median_cobra_time = ('avg_cobra_time', lambda x: x.quantile(0.50)),
     median_ocara_time = ('avg_ocara_time', lambda x: x.quantile(0.50)),
     
     ad_request = ('ad_request', 'sum'),
     ad_response = ('ad_response', 'sum'),
     ad_viewable_impression = ('ad_viewable_impression', 'sum'),
     ad_click = ('ad_click', 'sum'),
     
     pos_ad_request = ('pos_ad_request', 'sum'),
     pos_ad_response = ('pos_ad_response', 'sum'),
     pos_ad_viewable_impression = ('pos_ad_viewable_impression', 'sum'),
     pos_ad_click = ('pos_ad_click', 'sum'),
     
     ontheway_ad_request = ('ontheway_ad_request', 'sum'),
     ontheway_ad_response = ('ontheway_ad_response', 'sum'),
     ontheway_ad_viewable_impression = ('ontheway_ad_viewable_impression', 'sum'),
     ontheway_ad_click = ('ontheway_ad_click', 'sum'),
     
     arrived_ad_request = ('arrived_ad_request', 'sum'),
     arrived_ad_response = ('arrived_ad_response', 'sum'),
     arrived_ad_viewable_impression = ('arrived_ad_viewable_impression', 'sum'),
     arrived_ad_click = ('arrived_ad_click', 'sum'),
     
     started_ad_request = ('started_ad_request', 'sum'),
     started_ad_response = ('started_ad_response', 'sum'),
     started_ad_viewable_impression = ('started_ad_viewable_impression', 'sum'),
     started_ad_click = ('started_ad_click', 'sum'),
    )\
.reset_index()

df3

,pageVariant,city_name,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,median_cobrm_time,median_cobra_time,median_ocara_time,ad_request,ad_response,ad_viewable_impression,ad_click,pos_ad_request,pos_ad_response,pos_ad_viewable_impression,pos_ad_click,ontheway_ad_request,ontheway_ad_response,ontheway_ad_viewable_impression,ontheway_ad_click,arrived_ad_request,arrived_ad_response,arrived_ad_viewable_impression,arrived_ad_click,started_ad_request,started_ad_response,started_ad_viewable_impression,started_ad_click
0,l1Cancel,Bangalore,11667,5567,163,2491,2068,22,1312,83.500000,134.250000,338.00,51357.0,13625.0,3782.0,29.0,32187.0,13625.0,3782.0,29.0,14204.0,5811.0,1580.0,17.0,6569.0,2664.0,562.0,1.0,11432.0,5150.0,1640.0,11.0
1,l1Cancel,Chennai,417511,264622,4080,52444,72068,579,21946,110.000000,129.000000,313.25,3049440.0,804021.0,149662.0,1571.0,2037527.0,804021.0,149662.0,1571.0,988347.0,362322.0,73682.0,884.0,438010.0,165972.0,24729.0,72.0,612357.0,275727.0,51251.0,615.0
2,l1Cancel,Delhi,300001,154200,4101,49955,58512,377,31870,116.000000,136.000000,347.00,2181352.0,602414.0,130114.0,1117.0,1356379.0,602414.0,130114.0,1117.0,691856.0,295431.0,68699.0,617.0,294470.0,135498.0,23409.0,56.0,371112.0,171487.0,38006.0,444.0
3,l1Cancel,Hyderabad,917587,496146,5785,175679,153629,1206,81591,107.000000,147.000000,343.00,6770296.0,1286138.0,243588.0,2524.0,4384708.0,1286138.0,243588.0,2524.0,2037531.0,542791.0,120649.0,1555.0,1007175.0,247679.0,35659.0,124.0,1342891.0,495668.0,87280.0,845.0
4,l1Cancel,Jaipur,64213,29755,381,16404,12159,73,5253,132.000000,141.800000,380.00,464249.0,116226.0,23508.0,275.0,280988.0,116226.0,23508.0,275.0,135074.0,51532.0,11399.0,117.0,66067.0,28958.0,4811.0,16.0,79995.0,35736.0,7298.0,142.0
5,l1Cancel,Kolkata,2727,1477,6,275,776,7,177,172.000000,147.000000,341.50,10105.0,3029.0,1069.0,6.0,7184.0,3029.0,1069.0,6.0,3515.0,1372.0,509.0,2.0,1360.0,619.0,172.0,0.0,2315.0,1038.0,388.0,4.0
6,l1Cancel,Pune,67136,42511,285,7806,13181,42,3187,115.000000,125.416667,252.00,489021.0,150638.0,29305.0,218.0,335576.0,150638.0,29305.0,218.0,150970.0,66509.0,13632.0,128.0,72072.0,32251.0,5104.0,10.0,112769.0,51878.0,10569.0,80.0
7,l2Cancel,Bangalore,739148,344671,6989,178830,116440,850,89118,79.333333,132.000000,321.00,5308207.0,1352087.0,372315.0,3509.0,3234994.0,1352087.0,372315.0,3509.0,1546969.0,613967.0,168305.0,1695.0,694543.0,283035.0,72082.0,176.0,996262.0,455086.0,131929.0,1638.0
8,l2Cancel,Chennai,54913,35794,807,5636,10088,105,2196,100.000000,120.000000,329.00,467119.0,107546.0,25005.0,290.0,313119.0,107546.0,25005.0,290.0,162032.0,48506.0,11460.0,143.0,65414.0,21630.0,4399.0,23.0,85852.0,37410.0,9146.0,124.0
9,l2Cancel,Delhi,371620,194541,5255,60938,72006,578,36977,113.000000,134.000000,361.00,2755954.0,739862.0,223543.0,2066.0,1744479.0,739862.0,223543.0,2066.0,895122.0,360056.0,108332.0,1109.0,380112.0,166162.0,46902.0,129.0,470740.0,213644.0,68309.0,828.0
